# Inspect the HTB demo network-log dataset

Companion to HTB Academy Module 292, Section 5: **Datasets**. Run top to bottom with the AI kernel. This notebook only loads and inspects the exact course CSV. It does **not** clean, transform, split, or train on records; those operations belong to later lessons. Outputs are intentionally unsaved so example IP addresses and raw rows are not committed.

## Get the course file, then parse it

HTB links a ZIP archive containing demo_dataset.csv and shows pandas reading an already extracted CSV. The cell below is a labeled **teaching convenience**: use a local demo_dataset.zip beside this notebook if present; alternatively set HTB_DEMO_DATASET_ZIP to an existing ZIP path; otherwise download the small archive from HTB's content host. It reads the CSV inside the ZIP without extracting arbitrary archive paths. The archive member and uncompressed size are checked before parsing.

In [ ]:
from io import BytesIO
import os
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import requests

course_url = 'https://cdn.services-k8s.prod.aws.htb.systems/content/modules/292/demo_dataset.zip'
zip_path = Path(os.environ.get('HTB_DEMO_DATASET_ZIP', 'demo_dataset.zip'))
if zip_path.is_file():
    archive_bytes = zip_path.read_bytes()  # Reuse an already downloaded course archive.
    print('Using local course ZIP:', zip_path)
else:
    response = requests.get(course_url, timeout=30)  # Only the HTB course asset.
    response.raise_for_status()
    archive_bytes = response.content
    print('Downloaded HTB course ZIP in memory:', len(archive_bytes), 'bytes')

with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('demo_dataset.csv')  # Reject a missing or renamed member.
    if member.file_size > 10_000_000:
        raise ValueError('Unexpectedly large demo CSV; inspect the archive before parsing')
    with archive.open(member) as csv_file:
        data = pd.read_csv(csv_file)  # One DataFrame row per network-log event.

required = {'log_id', 'source_ip', 'destination_port', 'protocol',
            'bytes_transferred', 'threat_level'}
if not required.issubset(data.columns):
    raise ValueError(f'Unexpected columns: {sorted(data.columns)}')
print('DataFrame shape (rows, columns):', data.shape)


## What do the rows and column types look like?

head() previews the first five rows, including raw source IPs; do not paste real production logs into a public notebook. info() prints the inferred type and non-null count of each column. Its return value is None, so the notebook calls it without print(). A numeric-looking column may show object/string type when some rows contain words.

In [ ]:
print(data.head())     # The five-row preview HTB asks us to inspect.
data.info()          # Column names, inferred dtypes, and non-null counts.


## Actual nulls versus text sentinels

For each column, isnull() produces True for genuine pandas nulls and False otherwise. sum() counts the True values: [False, True, True] has a missing count of 2. A literal question mark or the word STRING_PORT can still count as **non-null**, so inspect suspicious values separately. Here we *report* them but do not replace or remove anything.

In [ ]:
print('True nulls per column:')
print(data.isnull().sum())

# A numeric parse check counts unexpected strings without changing the source table.
for column in ('destination_port', 'bytes_transferred'):
    as_number = pd.to_numeric(data[column], errors='coerce')
    unexpected_text = data[column].notna() & as_number.isna()
    print(column, 'non-null values that are not numeric:', int(unexpected_text.sum()))

print('Raw threat-level values, including unknown markers:')
print(data['threat_level'].value_counts(dropna=False))
assert len(data) > 0


An inspection report is a map of what needs attention, not a cleaned dataset. In the next sections, treat unknown labels and non-numeric fields explicitly, and fit any learned preprocessing only on training data. The course's 0/1/2 labels are categories here; they are not probabilities or calibrated risk scores.